# Série Temporal de Consumo Elétrico por CP7 — Aveiro (2024–2025)


1. Lê o ficheiro E-Redes original (com `Código Postal` por linha)
2. Projeta 2024–2025 **por CP7**, usando o perfil real de Fevereiro como âncora
3. Guarda uma série longa `(data_hora, cp7, energia_ativa_kwh)` pronta para cruzar com o Voronoi dos PTDs

**Inputs:**
- `consumoshorario_cp7_3800.csv` — consumos E-Redes dos CP7s com prefixo 3800
- `consumoshorario_cp7_3810.csv` — consumos E-Redes dos CP7s com prefixo 3810

**Output:** `serie_consumo_cp7_2024_2025.csv` — série horária por CP7, 2 anos (ambos os prefixos combinados)

In [1]:
# =============================================================
# CÉLULA 1 — IMPORTAÇÕES
# =============================================================
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("Bibliotecas importadas com sucesso.")

Bibliotecas importadas com sucesso.


## 1. Carregar e Concatenar os Dois Ficheiros E-Redes

In [2]:
# =============================================================
# CÉLULA 2 — CARREGAR E CONCATENAR OS DOIS FICHEIROS E-REDES
# =============================================================
# Os dois ficheiros têm estrutura idêntica — diferem apenas no prefixo do CP7
FICHEIROS_EREDES = [
    "consumoshorario_cp7_3800.csv",   # CPs começados por 3800
    "consumoshorario_cp7_3810.csv",   # CPs começados por 3810
]

def carregar_ficheiro(path):
    """Carrega um ficheiro E-Redes e normaliza colunas."""
    df = pd.read_csv(
        path,
        sep=";",             # separador ponto e vírgula
        encoding="utf-8-sig", # remove BOM (\ufeff) automaticamente
        parse_dates=["Data/Hora"],
        dayfirst=False
    )
    df = df.rename(columns={
        "Data/Hora"          : "data_hora",
        "Código Postal"      : "cp7",
        "Energia ativa (kWh)": "energia_kwh"
    })
    df["data_hora"] = pd.to_datetime(df["data_hora"], utc=True).dt.tz_localize(None)
    return df[["data_hora", "cp7", "energia_kwh"]].copy()

# Carregar e concatenar os dois ficheiros
partes = []
for f in FICHEIROS_EREDES:
    df_parte = carregar_ficheiro(f)
    print(f"  {f}: {len(df_parte):,} registos | {df_parte['cp7'].nunique()} CP7s")
    partes.append(df_parte)

df_raw = pd.concat(partes, ignore_index=True)

# Verificar duplicados (mesmo CP7 + data_hora nos dois ficheiros)
n_antes = len(df_raw)
df_raw = df_raw.drop_duplicates(subset=["data_hora", "cp7"])
n_dup = n_antes - len(df_raw)
if n_dup > 0:
    print(f"  ⚠️  {n_dup} linhas duplicadas removidas (mesmo CP7 + hora nos dois ficheiros)")

df_raw = df_raw.sort_values(["cp7", "data_hora"]).reset_index(drop=True)

# Colunas auxiliares temporais
df_raw["mes"]  = df_raw["data_hora"].dt.month
df_raw["dia"]  = df_raw["data_hora"].dt.day
df_raw["hora"] = df_raw["data_hora"].dt.hour

lista_cp7 = sorted(df_raw["cp7"].unique())

print(f"\n--- Totais após junção ---")
print(f"Registos totais       : {len(df_raw):,}")
print(f"Códigos postais únicos: {len(lista_cp7)}")
print(f"  CP7s 3800-xxx       : {sum(1 for c in lista_cp7 if c.startswith('3800'))}")
print(f"  CP7s 3810-xxx       : {sum(1 for c in lista_cp7 if c.startswith('3810'))}")
print(f"Janela temporal       : {df_raw['data_hora'].min()} → {df_raw['data_hora'].max()}")
print(f"Meses presentes       : {sorted(df_raw['mes'].unique())}")
print(f"\nPrimeiras linhas:")
print(df_raw.head(5).to_string(index=False))

  consumoshorario_cp7_3800.csv: 241,341 registos | 369 CP7s
  consumoshorario_cp7_3810.csv: 238,038 registos | 352 CP7s

--- Totais após junção ---
Registos totais       : 479,379
Códigos postais únicos: 721
  CP7s 3800-xxx       : 369
  CP7s 3810-xxx       : 352
Janela temporal       : 2024-02-01 00:00:00 → 2024-02-29 23:00:00
Meses presentes       : [np.int32(2)]

Primeiras linhas:
          data_hora      cp7  energia_kwh  mes  dia  hora
2024-02-01 00:00:00 3800-003   181.585557    2    1     0
2024-02-01 01:00:00 3800-003   141.137459    2    1     1
2024-02-01 02:00:00 3800-003   125.353439    2    1     2
2024-02-01 03:00:00 3800-003   130.794638    2    1     3
2024-02-01 04:00:00 3800-003   120.499604    2    1     4


## 2. Diagnóstico de Cobertura por CP7

Verifica quantas horas de dados cada CP7 tem em Fevereiro (o mês âncora).
CP7s com menos de 200 horas de dados serão tratados com o perfil médio de todos os CP7s como fallback.

In [3]:
# =============================================================
# CÉLULA 3 — DIAGNÓSTICO DE COBERTURA
# =============================================================
df_fev = df_raw[df_raw["mes"] == 2].copy()

cobertura = (
    df_fev.groupby("cp7")["energia_kwh"]
    .agg(n_horas="count", consumo_total_kwh="sum", media_hora_kwh="mean")
    .sort_values("consumo_total_kwh", ascending=False)
    .reset_index()
)

MIN_HORAS = 200  # mínimo para usar perfil próprio (fev tem 672h = 28 dias × 24h)
cp7_bons    = cobertura[cobertura["n_horas"] >= MIN_HORAS]["cp7"].tolist()
cp7_escassos = cobertura[cobertura["n_horas"] <  MIN_HORAS]["cp7"].tolist()

print(f"Total CP7s                         : {len(lista_cp7)}")
print(f"Com ≥{MIN_HORAS}h em Fev (perfil próprio): {len(cp7_bons)}")
print(f"Com <{MIN_HORAS}h (usam perfil médio)    : {len(cp7_escassos)}")
print(f"\nTop 10 CP7s por consumo total em Fevereiro:")
print(cobertura.head(10).to_string(index=False))

Total CP7s                         : 721
Com ≥200h em Fev (perfil próprio): 700
Com <200h (usam perfil médio)    : 21

Top 10 CP7s por consumo total em Fevereiro:
     cp7  n_horas  consumo_total_kwh  media_hora_kwh
3800-536      696       3.777615e+06     5427.607241
3800-055      696       2.460727e+06     3535.527675
3800-587      696       1.363488e+06     1959.035020
3810-498      696       1.141867e+06     1640.613305
3810-140      696       8.396468e+05     1206.389063
3800-525      676       8.177344e+05     1209.666246
3810-168      696       7.750704e+05     1113.606861
3810-783      696       7.493012e+05     1076.582192
3810-434      696       6.190779e+05      889.479718
3810-106      696       5.204202e+05      747.730110


## 3. Construir Perfis Horários de Fevereiro por CP7

Para cada CP7, calcula o consumo médio por `(dia_do_mês, hora)` em Fevereiro.
Este perfil é o **molde base** para toda a projeção.

In [4]:
# =============================================================
# CÉLULA 4 — PERFIS HORÁRIOS POR CP7
# =============================================================

# Perfil médio global (fallback para CP7s com dados escassos)
perfil_global = (
    df_fev.groupby(["dia", "hora"])["energia_kwh"]
    .mean()
    .to_dict()
)
# Fallback por hora apenas (se também não houver o par dia+hora)
perfil_global_hora = (
    df_fev.groupby("hora")["energia_kwh"]
    .mean()
    .to_dict()
)

# Perfil por CP7 — dicionário: cp7 → {(dia, hora): valor_médio}
perfis_cp7 = {}
for cp in lista_cp7:
    df_cp_fev = df_fev[df_fev["cp7"] == cp]
    perfis_cp7[cp] = (
        df_cp_fev.groupby(["dia", "hora"])["energia_kwh"]
        .mean()
        .to_dict()
    )

print(f"Perfis construídos para {len(perfis_cp7)} CP7s.")
print("Exemplo — entradas do primeiro CP7:", list(list(perfis_cp7.values())[0].items())[:4])

Perfis construídos para 721 CP7s.
Exemplo — entradas do primeiro CP7: [((1, 0), 181.58555657730088), ((1, 1), 141.13745899944223), ((1, 2), 125.35343928993744), ((1, 3), 130.79463792118565)]


## 4. Coeficientes Sazonais Mensais

Fevereiro = 1.00 (mês âncora — dados reais medidos).  
Os restantes meses são escalados com base em padrões sazonais para Aveiro.

In [5]:
# =============================================================
# CÉLULA 5 — COEFICIENTES SAZONAIS
# =============================================================
coeficientes = {
     1: 1.25,   # Janeiro  — Inverno rigoroso; aquecimento elevado
     2: 1.00,   # Fevereiro — MÊS ÂNCORA (dados reais E-Redes)
     3: 1.05,   # Março    — Fim do inverno, aquecimento ainda presente
     4: 0.95,   # Abril    — Primavera temperada
     5: 0.85,   # Maio     — Luz natural reduz iluminação artificial
     6: 0.75,   # Junho    — Menor atividade escolar/universitária
     7: 0.70,   # Julho    — Época balnear, férias, menos residentes
     8: 0.65,   # Agosto   — Mínimo anual; empresas encerradas
     9: 0.80,   # Setembro — Retorno escolar e industrial
    10: 0.95,   # Outubro  — Início de climatização pontual
    11: 1.15,   # Novembro — Início do período crítico de frio
    12: 1.30,   # Dezembro — Máximo anual: noites longas + frio
}

# Crescimento estrutural de +2% em 2025 (ERSE / REN)
FATOR_2025 = 1.02

print("Coeficientes sazonais:")
for m, c in coeficientes.items():
    flag = " ← âncora" if m == 2 else ""
    print(f"  Mês {m:2d}: {c:.2f}{flag}")

Coeficientes sazonais:
  Mês  1: 1.25
  Mês  2: 1.00 ← âncora
  Mês  3: 1.05
  Mês  4: 0.95
  Mês  5: 0.85
  Mês  6: 0.75
  Mês  7: 0.70
  Mês  8: 0.65
  Mês  9: 0.80
  Mês 10: 0.95
  Mês 11: 1.15
  Mês 12: 1.30


## 5. Projeção Horária 2024–2025 por CP7

**Lógica por hora de cada CP7:**
- Jan e Fev 2024: usar valor real do ficheiro E-Redes (se existir)
- Restante 2024 e todo o 2025: perfil de Fev do CP7 × coeficiente sazonal [× 1.02 se 2025]
- Dias 29/30/31 sem equivalente em Fev: usa o dia 28 como proxy
- CP7 sem perfil suficiente: usa perfil médio global como fallback

In [6]:
# =============================================================
# CÉLULA 6 — PROJEÇÃO POR CP7 (vectorizada)
# =============================================================

# Calendário horário completo 2024–2025
horas = pd.date_range(start="2024-01-01", end="2025-12-31 23:00:00", freq="h")

# Índice rápido dos dados reais: (cp7, mes, dia, hora) → energia_kwh
df_raw_idx = df_raw.set_index(["cp7", "mes", "dia", "hora"])["energia_kwh"]

resultados = []
total_cp7  = len(lista_cp7)

for i, cp in enumerate(lista_cp7):
    if (i + 1) % 20 == 0 or i == total_cp7 - 1:
        print(f"  Processados {i+1}/{total_cp7} CP7s...", end="\r")

    perfil = perfis_cp7[cp]  # {(dia, hora): valor}
    usar_global = cp in cp7_escassos

    for ts in horas:
        ano_  = ts.year
        mes_  = ts.month
        dia_  = ts.day
        hora_ = ts.hour

        energia = None

        # Jan e Fev 2024: dados reais primeiro
        if ano_ == 2024 and mes_ in (1, 2):
            try:
                energia = float(df_raw_idx.loc[(cp, mes_, dia_, hora_)])
            except KeyError:
                pass  # cai para o perfil abaixo

        if energia is None:
            # Mapear dias sem equivalente em Fevereiro para dia 28
            dia_fev = min(dia_, 28)

            if usar_global:
                consumo_base = (
                    perfil_global.get((dia_fev, hora_))
                    or perfil_global_hora.get(hora_, 0.0)
                )
            else:
                consumo_base = (
                    perfil.get((dia_fev, hora_))
                    or perfil_global.get((dia_fev, hora_))
                    or perfil_global_hora.get(hora_, 0.0)
                )

            coef        = coeficientes[mes_]
            fator_anual = FATOR_2025 if ano_ == 2025 else 1.0
            energia     = consumo_base * coef * fator_anual

        resultados.append((ts, cp, round(energia, 6)))

print(f"\nProjeção concluída: {len(resultados):,} registos horários.")

  Processados 721/721 CP7s...
Projeção concluída: 12,649,224 registos horários.


## 6. Montar e Guardar o DataFrame Final

In [7]:
# =============================================================
# CÉLULA 7 — GUARDAR SÉRIE POR CP7
# =============================================================
df_serie = pd.DataFrame(resultados, columns=["data_hora", "cp7", "energia_ativa_kwh"])

# Colunas auxiliares úteis para análises posteriores
df_serie["ano"]           = df_serie["data_hora"].dt.year
df_serie["mes"]           = df_serie["data_hora"].dt.month
df_serie["dia"]           = df_serie["data_hora"].dt.day
df_serie["hora"]          = df_serie["data_hora"].dt.hour
df_serie["dia_da_semana"] = df_serie["data_hora"].dt.dayofweek   # 0=Seg, 6=Dom
df_serie["chave_mes_ano"] = df_serie["data_hora"].dt.strftime("%Y-%m")
df_serie["cp4"]           = df_serie["cp7"].str.split("-").str[0]  # primeiros 4 dígitos

OUTPUT = "serie_consumo_cp7_2024_2025.csv"
df_serie.to_csv(OUTPUT, index=False)

print("=" * 55)
print(f"Ficheiro guardado  : {OUTPUT}")
print(f"Total de linhas    : {len(df_serie):,}")
print(f"CP7s na série      : {df_serie['cp7'].nunique()}")
print(f"Janela temporal    : {df_serie['data_hora'].min()} → {df_serie['data_hora'].max()}")
print(f"Colunas            : {list(df_serie.columns)}")
print("=" * 55)

Ficheiro guardado  : serie_consumo_cp7_2024_2025.csv
Total de linhas    : 12,649,224
CP7s na série      : 721
Janela temporal    : 2024-01-01 00:00:00 → 2025-12-31 23:00:00
Colunas            : ['data_hora', 'cp7', 'energia_ativa_kwh', 'ano', 'mes', 'dia', 'hora', 'dia_da_semana', 'chave_mes_ano', 'cp4']


## 7. Validação — Totais Mensais por CP7

Verifica se a distribuição sazonal faz sentido: Dezembro deve ser o mês mais alto, Agosto o mais baixo.

In [8]:
# =============================================================
# CÉLULA 8 — VALIDAÇÃO: TOTAIS MENSAIS AGREGADOS
# =============================================================
resumo_mensal = (
    df_serie.groupby(["ano", "mes"])["energia_ativa_kwh"]
    .sum()
    .reset_index()
    .rename(columns={"energia_ativa_kwh": "total_kwh"})
)
resumo_mensal["total_MWh"] = (resumo_mensal["total_kwh"] / 1000).round(1)

print("Consumo total por mês (todos os CP7s):")
print(resumo_mensal.to_string(index=False))

# Top 10 CP7s por consumo anual em 2024
print("\nTop 10 CP7s — consumo total 2024:")
top_cp7 = (
    df_serie[df_serie["ano"] == 2024]
    .groupby("cp7")["energia_ativa_kwh"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top_cp7["total_MWh"] = (top_cp7["energia_ativa_kwh"] / 1000).round(1)
print(top_cp7[["cp7", "total_MWh"]].to_string(index=False))

Consumo total por mês (todos os CP7s):
 ano  mes    total_kwh  total_MWh
2024    1 5.565433e+07    55654.3
2024    2 4.139226e+07    41392.3
2024    3 4.674963e+07    46749.6
2024    4 4.084791e+07    40847.9
2024    5 3.784494e+07    37844.9
2024    6 3.224835e+07    32248.3
2024    7 3.116642e+07    31166.4
2024    8 2.894025e+07    28940.3
2024    9 3.439824e+07    34398.2
2024   10 4.229729e+07    42297.3
2024   11 4.944747e+07    49447.5
2024   12 5.788050e+07    57880.5
2025    1 5.676741e+07    56767.4
2025    2 4.074540e+07    40745.4
2025    3 4.768463e+07    47684.6
2025    4 4.166487e+07    41664.9
2025    5 3.860184e+07    38601.8
2025    6 3.289332e+07    32893.3
2025    7 3.178975e+07    31789.8
2025    8 2.951906e+07    29519.1
2025    9 3.508620e+07    35086.2
2025   10 4.314323e+07    43143.2
2025   11 5.043642e+07    50436.4
2025   12 5.903811e+07    59038.1

Top 10 CP7s — consumo total 2024:
     cp7  total_MWh
3800-536    44918.4
3800-055    29630.6
3800-587    1644

## 8. Visualização — Consumo Mensal por CP7 (Top 10)

In [9]:
# =============================================================
# CÉLULA 9 — GRÁFICO: SÉRIE MENSAL AGREGADA 2024–2025
# =============================================================
resumo_mes_str = resumo_mensal.copy()
resumo_mes_str["periodo"] = resumo_mes_str["ano"].astype(str) + "-" + resumo_mes_str["mes"].astype(str).str.zfill(2)

fig_total = go.Figure()
fig_total.add_trace(go.Bar(
    x=resumo_mes_str["periodo"],
    y=resumo_mes_str["total_MWh"],
    marker_color=["#0284C7" if a == 2024 else "#0D9488" for a in resumo_mes_str["ano"]],
    name="Consumo (MWh)"
))
fig_total.update_layout(
    title="Consumo Total Mensal — Aveiro (todos os CP7s)",
    xaxis_title="Mês", yaxis_title="Energia (MWh)",
    plot_bgcolor="white", paper_bgcolor="white",
    height=400
)
fig_total.update_xaxes(tickangle=45, showgrid=False)
fig_total.update_yaxes(showgrid=True, gridcolor="#F1F5F9")
fig_total.show()

# --- Top 10 CP7s: perfil horário médio em Fevereiro ---
top10_cp7 = top_cp7["cp7"].tolist()
df_top_fev = df_serie[
    (df_serie["cp7"].isin(top10_cp7)) &
    (df_serie["mes"] == 2) &
    (df_serie["ano"] == 2024)
]
perfil_top = df_top_fev.groupby(["cp7", "hora"])["energia_ativa_kwh"].mean().reset_index()

fig_perfil = go.Figure()
cores = ["#0284C7","#0D9488","#7C3AED","#DC2626","#EA580C",
         "#D97706","#65A30D","#0891B2","#9333EA","#DB2777"]
for j, cp in enumerate(top10_cp7):
    sub = perfil_top[perfil_top["cp7"] == cp]
    fig_perfil.add_trace(go.Scatter(
        x=sub["hora"], y=sub["energia_ativa_kwh"],
        mode="lines", name=cp,
        line=dict(color=cores[j % len(cores)], width=1.8)
    ))
fig_perfil.update_layout(
    title="Perfil Horário Médio — Fevereiro 2024 (Top 10 CP7s)",
    xaxis_title="Hora", yaxis_title="Energia média (kWh)",
    plot_bgcolor="white", paper_bgcolor="white",
    height=420, legend_title="CP7"
)
fig_perfil.update_xaxes(tickvals=list(range(0, 24, 2)), showgrid=False)
fig_perfil.update_yaxes(showgrid=True, gridcolor="#F1F5F9")
fig_perfil.show()

In [10]:
# =============================================================
# CÉLULA 10 — PRÉ-VISUALIZAÇÃO DO FICHEIRO FINAL
# =============================================================
print("Estrutura do ficheiro de saída:")
print(df_serie[["data_hora","cp7","cp4","energia_ativa_kwh","ano","mes","hora"]].head(10).to_string(index=False))
print(f"\nFicheiro pronto: serie_consumo_cp7_2024_2025.csv")
print("Próximo passo  : carregar este CSV no notebook de análise CER por PTD.")

Estrutura do ficheiro de saída:
          data_hora      cp7  cp4  energia_ativa_kwh  ano  mes  hora
2024-01-01 00:00:00 3800-003 3800         226.981946 2024    1     0
2024-01-01 01:00:00 3800-003 3800         176.421824 2024    1     1
2024-01-01 02:00:00 3800-003 3800         156.691799 2024    1     2
2024-01-01 03:00:00 3800-003 3800         163.493297 2024    1     3
2024-01-01 04:00:00 3800-003 3800         150.624505 2024    1     4
2024-01-01 05:00:00 3800-003 3800         153.525284 2024    1     5
2024-01-01 06:00:00 3800-003 3800         172.675324 2024    1     6
2024-01-01 07:00:00 3800-003 3800         221.477052 2024    1     7
2024-01-01 08:00:00 3800-003 3800         251.570741 2024    1     8
2024-01-01 09:00:00 3800-003 3800         234.281900 2024    1     9

Ficheiro pronto: serie_consumo_cp7_2024_2025.csv
Próximo passo  : carregar este CSV no notebook de análise CER por PTD.
